# 03 — Embeddings & Semantic Recall (RAG)

An **embedding** turns text into a vector so that *similar meaning* lands *nearby* in
vector space. That's the engine behind semantic memory (L5 / E1) and RAG.

This notebook runs fully offline using Vigil's deterministic local embedding. For
*real* semantic similarity, set `VIGIL_EMBED_MODE=openai` (needs an API key).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from levels.l5_memory import _embed_text_local, _cosine

In [ ]:
docs = [
    'Remote code execution in Apache Log4j via JNDI lookup.',
    'Log4Shell lets attackers run arbitrary code through crafted log messages.',
    'A cross-site scripting bug in a web form input field.',
]
vectors = [_embed_text_local(d) for d in docs]
print('vector dimensions:', len(vectors[0]))

In [ ]:
query = 'arbitrary code execution in Log4j'
qv = _embed_text_local(query)
ranked = sorted(((round(_cosine(qv, v), 3), d) for v, d in zip(vectors, docs)), reverse=True)
for score, d in ranked:
    print(f'{score:>6}  {d}')

## Local vs real embeddings
The **local** embedding is a deterministic hash — great for offline, reproducible demos,
but it can't capture true semantics (synonyms won't necessarily score higher).
Switching `VIGIL_EMBED_MODE=openai` swaps in `text-embedding-3-small` (1536-d), where
'arbitrary code execution' and 'remote code execution' genuinely land near each other.

See `embed_text`, `_embed_text_local`, and `_embed_text_openai` in `levels/l5_memory.py`.